In [ ]:
import glob
import os
from multiprocessing import Process

import pandas as pd
import yaml

from pynxtools_em.examples.oasisb.batch_process import process_project
from pynxtools_em.examples.oasisb.oasisb_utils import get_project_id

print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory: str = f"{fp.readline().strip().replace('/', os.sep)}"
print(src_directory)
with open("target_directory.txt") as fp:
    trg_directory: str = f"{fp.readline().strip().replace('/', os.sep)}"
print(trg_directory)
with open("alias_prefix_secret.txt") as fp:
    alias_prefix_secret: str = f"{fp.readline().strip().replace('/', os.sep)}"
print(alias_prefix_secret)

os.makedirs(f"{trg_directory.replace('/decompressed', '/pynxtools')}", exist_ok=True)
# os.listdir(f"{trg_directory.replace('/decompressed', '')}")

In [ ]:
spread_sheet_of_all_projects = pd.read_excel(
    f"{trg_directory.replace('/decompressed', '/config')}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype=str,
).fillna("")

project_range: tuple[int, int] = (302, 400)

with open(
    f"{trg_directory.replace('/decompressed', '/config')}{os.sep}aaa_em_nomad_project_names.yaml"
) as fp:
    nomad_project_names: dict[str, str] = yaml.safe_load(fp)

Run the processing queue generating NeXus/HDF5 files.

In [ ]:
count: int = 0
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        project_id = get_project_id(row.project_name)
        if project_range[0] <= int(project_id) <= project_range[1]:
            if row.legal == "1" and row.use == "1":
                if project_id in nomad_project_names:
                    # if project_id not in (""):
                    #     continue
                    for mime_type in ["msa", "dm3", "dm4", "emd"]:

                        def work_package():  # assure memory gets returned to the operating system
                            process_project(
                                project_id,
                                # f"{src_directory}{os.sep}{project_id}.ods",
                                f"{trg_directory.replace('/decompressed', '/config')}{os.sep}aaa_legacy_data.bib",
                                "",  # no file_path_aliasing f"{trg_directory}{os.sep}{project_id}.mixed.decompressed.csv",
                                trg_directory,
                                f"{trg_directory.replace('/decompressed', '/pynxtools')}",
                                mime_type,  # "msa",
                                alias_prefix_secret,
                                openalex_file=f"{os.getcwd()}{os.sep}openalex/D{project_id}.json",
                                logger_file_path_suffix=mime_type,  # "msa",
                                nomad_project_name=nomad_project_names[project_id],
                            )

                        p = Process(target=work_package)
                        p.start()
                        p.join()
                count += 1
print(f"Batch queue completed {count}")

Check if each mtex.h5 file has a matching nxs file.

In [ ]:
count = 0
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        project_id = get_project_id(row.project_name)
        if project_range[0] <= int(project_id) <= project_range[1]:
            if row.legal == "1" and row.use == "1":
                if project_id in nomad_project_names:
                    pattern = os.path.join(trg_directory, f"{project_id}.*.mtex.h5")
                    preprocessed_hfive_files: list[str] = glob.glob(pattern)
                    for hfive_file in preprocessed_hfive_files:
                        file_name = hfive_file.rsplit(os.sep, 1)[1]
                        output_file_path = f"{trg_directory}{os.sep}..{os.sep}pynxtools{os.sep}{file_name}.nxs"
                        if not os.path.isfile(output_file_path):
                            print(output_file_path)
                            count += 1
                    del pattern, preprocessed_hfive_files
print(f"Batch queue completed {count}")

***